In [160]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Read data from fz1 - FZ1.1

In [161]:
# First read lines 8 and 9 to check which one contains the header
header_check = pd.read_excel('../../Data prep/Bestand/01 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Zulassungsbezirken (FZ 1)/fz1_2025.xlsx', 
                            sheet_name='FZ1.1', 
                            skiprows=7, 
                            nrows=2)

In [162]:
# Check if line 8 (first row) is empty
if header_check.iloc[1].isna().all():
    header_row = 8  # Use line 9 as header
else:
    header_row = 7  # Use line 8 as header

print(f"Using row {header_row + 1} as header")

Using row 8 as header


In [163]:
# Now read the full file with the correct header row
df = pd.read_excel('../../Data prep/Bestand/01 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Zulassungsbezirken (FZ 1)/fz1_2025.xlsx', 
                   sheet_name='FZ1.1', 
                   header=header_row)


In [164]:
df = df.iloc[:, 1:]

In [165]:
# Check if there are any unnamed columns
if df.columns.str.contains('^Unnamed').any():
    first_row = df.iloc[0]

    # Replace only the unnamed headers
    new_columns = [
        first_row[i] if col.startswith('Unnamed') else col
        for i, col in enumerate(df.columns)
    ]

    df.columns = new_columns
    df = df[1:].reset_index(drop=True)  # Drop the first row (used as partial header)

C:\Users\janta\AppData\Local\Temp\ipykernel_17060\1403245063.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_row[i] if col.startswith('Unnamed') else col


In [166]:
# Forward fill multiple columns that might have merged cells
columns_to_fill = ['Land', 'Regierungsbezirk']  # Add any other columns that have merged cells
for col in columns_to_fill:
    df[col] = df[col].fillna(method='ffill')

# Display the results
# print("First 10 rows of the DataFrame:")
# print(df[columns_to_fill + ['Unnamed: 3']].head(10))

C:\Users\janta\AppData\Local\Temp\ipykernel_17060\1868915817.py:4: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[col] = df[col].fillna(method='ffill')


In [167]:
# Remove rows where column 1 contains "zusammen"
df = df[~df.iloc[:, 1].astype(str).str.contains('zusammen', case=False, na=False)].reset_index(drop=True)


In [170]:
df.columns = df.columns.str.replace('\n', ' ', regex=False)
df.columns = df.columns.str.strip()

In [171]:
df.head()

,Land,Regierungsbezirk,Statistische Kennziffer und Zulassungsbezirk,Krafträder,davon zweirädrige Kfz,davon dreirädrige Kfz,davon leichte vierrädrige Kfz,darunter Halterinnen,Personenkraftwagen,Hubraum bis 1.399 cm³,...,unbe- kannt,Zugmaschinen,davon Sattelzug- maschinen,davon land-/forst- wirtschaft- liche Zug- maschinen,davon sonstige Zug- maschinen,Sonstige Kfz,Nutzfahr- zeuge insgesamt,Kraftfahrzeuge,Kfz-Dichte je 1.000 Einwohner,Kraftfahrzeug- anhänger
0,BADEN-WUERTTEMBERG,RB STUTTGART,"08111 STUTTGART,STADT",28548,27812,537,199,3789,301985,93124,...,9,3131,798,1906,427,2590.0,24545.0,355078,561,19716.0
1,BADEN-WUERTTEMBERG,RB STUTTGART,08115 BOEBLINGEN,26719,26075,393,251,3591,265649,84848,...,7,7938,508,5306,2124,946.0,22537.0,314905,785,31448.0
2,BADEN-WUERTTEMBERG,RB STUTTGART,08116 ESSLINGEN,40747,39889,526,332,6221,345526,120137,...,10,11518,1077,6969,3472,1531.0,34321.0,420594,775,45978.0
3,BADEN-WUERTTEMBERG,RB STUTTGART,08117 GOEPPINGEN,19795,19324,229,242,3074,171708,60995,...,4,8579,477,5860,2242,723.0,20923.0,212426,806,27297.0
4,BADEN-WUERTTEMBERG,RB STUTTGART,08118 LUDWIGSBURG,38394,37559,528,307,5456,342151,118705,...,15,11949,1314,7903,2732,1460.0,35024.0,415569,751,43395.0


In [172]:
# Divide columns if needed
df[['Statistische Kennziffer', 'Zulassungsbezirk']] = df['Statistische Kennziffer und Zulassungsbezirk'].str.extract(r'(\d{5})\s+(.+)')

In [173]:
# Divide columns if needed
df['Regierungs_Bezirk'] = df['Regierungsbezirk'].str.replace(r'^[A-ZÄÖÜa-zäöü]{2}\s+', '', regex=True)


In [174]:
df.head()

,Land,Regierungsbezirk,Statistische Kennziffer und Zulassungsbezirk,Krafträder,davon zweirädrige Kfz,davon dreirädrige Kfz,davon leichte vierrädrige Kfz,darunter Halterinnen,Personenkraftwagen,Hubraum bis 1.399 cm³,...,davon land-/forst- wirtschaft- liche Zug- maschinen,davon sonstige Zug- maschinen,Sonstige Kfz,Nutzfahr- zeuge insgesamt,Kraftfahrzeuge,Kfz-Dichte je 1.000 Einwohner,Kraftfahrzeug- anhänger,Statistische Kennziffer,Zulassungsbezirk,Regierungs_Bezirk
0,BADEN-WUERTTEMBERG,RB STUTTGART,"08111 STUTTGART,STADT",28548,27812,537,199,3789,301985,93124,...,1906,427,2590.0,24545.0,355078,561,19716.0,08111,"STUTTGART,STADT",STUTTGART
1,BADEN-WUERTTEMBERG,RB STUTTGART,08115 BOEBLINGEN,26719,26075,393,251,3591,265649,84848,...,5306,2124,946.0,22537.0,314905,785,31448.0,08115,BOEBLINGEN,STUTTGART
2,BADEN-WUERTTEMBERG,RB STUTTGART,08116 ESSLINGEN,40747,39889,526,332,6221,345526,120137,...,6969,3472,1531.0,34321.0,420594,775,45978.0,08116,ESSLINGEN,STUTTGART
3,BADEN-WUERTTEMBERG,RB STUTTGART,08117 GOEPPINGEN,19795,19324,229,242,3074,171708,60995,...,5860,2242,723.0,20923.0,212426,806,27297.0,08117,GOEPPINGEN,STUTTGART
4,BADEN-WUERTTEMBERG,RB STUTTGART,08118 LUDWIGSBURG,38394,37559,528,307,5456,342151,118705,...,7903,2732,1460.0,35024.0,415569,751,43395.0,08118,LUDWIGSBURG,STUTTGART


## Read data from fz1 - FZ1.2

In [ ]:
# First read lines 8 and 9 to check which one contains the header
header_check = pd.read_excel('../../Data prep/Bestand/01 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Zulassungsbezirken (FZ 1)/fz1_2025.xlsx', 
                            sheet_name='FZ1.2', 
                            skiprows=7, 
                            nrows=2)

In [ ]:
# Check if line 8 (first row) is empty
if header_check.iloc[1].isna().all():
    header_row = 8  # Use line 9 as header
else:
    header_row = 7  # Use line 8 as header

# print(f"Using row {header_row + 1} as header")

In [ ]:
# Now read the full file with the correct header row
df_2 = pd.read_excel('../../Data prep/Bestand/01 Kraftfahrzeuge und Kraftfahrzeuganhänger nach Zulassungsbezirken (FZ 1)/fz1_2025.xlsx', 
                   sheet_name='FZ1.2', 
                   header=header_row)


In [ ]:
df_2

In [ ]:
df_2 = df_2.iloc[:, 1:]

In [ ]:
# Check if there are any unnamed columns
if df_2.columns.str.contains('^Unnamed').any():
    first_row = df.iloc[0]

    # Replace only the unnamed headers
    new_columns = [
        first_row[i] if col.startswith('Unnamed') else col
        for i, col in enumerate(df.columns)
    ]

    df_2.columns = new_columns
    df_2 = df_2[1:].reset_index(drop=True)  # Drop the first row (used as partial header)

In [ ]:
df_2

In [ ]:
df_2.columns = df_2.columns.str.replace('\n', ' ', regex=False)
df_2.columns = df_2.columns.str.replace('\t', ' ', regex=False)
df_2.columns = df_2.columns.str.strip()
# Check the columns
# print(df.columns.tolist())

In [ ]:
# Forward fill multiple columns that might have merged cells
columns_to_fill = ['Land', 'Regierungsbezirk']  # Add any other columns that have merged cells
for col in columns_to_fill:
    df[col] = df[col].fillna(method='ffill')

# Display the results
# print("First 10 rows of the DataFrame:")
# print(df[columns_to_fill + ['Unnamed: 3']].head(10))

In [ ]:
# Remove rows where column 1 contains "zusammen"
df = df[~df.iloc[:, 1].astype(str).str.contains('zusammen', case=False, na=False)].reset_index(drop=True)


In [ ]:
df.head()

In [ ]:
# Divide columns if needed
df[['Statistische Kennziffer', 'Zulassungsbezirk']] = df['Statistische Kennziffer und Zulassungsbezirk'].str.extract(r'(\d{5})\s+(.+)')

In [ ]:
# Divide columns if needed
df['Regierungs_Bezirk'] = df['Regierungsbezirk'].str.replace(r'^[A-ZÄÖÜa-zäöü]{2}\s+', '', regex=True)


In [ ]:
df.head()